In [7]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. 데이터 불러오기
# ============================================================

df = pd.read_csv("한전KDN(주)_CHAON 전기차 고장 데이터_20260903.csv")

print("원본 데이터 개수:", len(df))


# ============================================================
# 2. 시간 데이터 변환
# ============================================================

df["발생시간"] = pd.to_datetime(
    df["발생시간"],
    errors="coerce"
)

df["해소시간"] = pd.to_datetime(
    df["해소시간"],
    errors="coerce"
)


# 발생시간이 이상한 데이터 제거
df = df.dropna(subset=["발생시간"])


# ============================================================
# 3. 중복 제거
#
# 같은 충전기 + 같은 고장 + 같은 발생시간이면
# 하나의 고장으로 처리
# ============================================================

df = df.drop_duplicates(
    subset=[
        "충전기명",
        "고장코드번호",
        "발생시간"
    ]
)


# 시간순 정렬
df = df.sort_values("발생시간").reset_index(drop=True)

print("중복 제거 후:", len(df))


# ============================================================
# 4. 기본 시간 Feature 만들기
# ============================================================

df["시간"] = df["발생시간"].dt.hour

# 월=0 화=1 ... 일=6
df["요일"] = df["발생시간"].dt.weekday

df["월"] = df["발생시간"].dt.month


# ============================================================
# 5. 머신러닝용 데이터 생성
#
# 핵심:
#
# 현재 고장이 발생한 시점을 기준으로
#
# 최근 1일 고장 횟수
# 최근 7일 고장 횟수
# 최근 30일 고장 횟수
# 이전 고장 이후 지난 시간
#
# 을 계산한다.
#
# 그리고
#
# "현재 고장 이후 24시간 안에
#  또 고장이 발생했는가?"
#
# 를 정답 Y로 만든다.
# ============================================================

machine_data = []


# 충전기별로 따로 계산
for charger_name, group in df.groupby("충전기명"):

    group = group.sort_values("발생시간").reset_index(drop=True)

    times = group["발생시간"]


    # 해당 충전기의 각 고장 기록을 하나씩 확인
    for i in range(len(group)):

        current_time = times.iloc[i]


        # ----------------------------------------------------
        # 과거 데이터
        # ----------------------------------------------------

        past = times.iloc[:i]


        # 최근 1일
        count_1d = (
            past >= current_time - pd.Timedelta(days=1)
        ).sum()


        # 최근 7일
        count_7d = (
            past >= current_time - pd.Timedelta(days=7)
        ).sum()


        # 최근 30일
        count_30d = (
            past >= current_time - pd.Timedelta(days=30)
        ).sum()


        # ----------------------------------------------------
        # 이전 고장 이후 몇 시간이 지났는지
        # ----------------------------------------------------

        if i > 0:

            previous_time = times.iloc[i - 1]

            hours_since_error = (
                current_time - previous_time
            ).total_seconds() / 3600

        else:

            hours_since_error = 9999


        # ----------------------------------------------------
        # 정답 Y
        #
        # 현재 고장 이후 다음 고장이
        # 24시간 안에 발생하면 1
        #
        # 아니면 0
        # ----------------------------------------------------

        if i + 1 < len(group):

            next_time = times.iloc[i + 1]

            hours_until_next = (
                next_time - current_time
            ).total_seconds() / 3600


            if hours_until_next <= 24:
                target = 1
            else:
                target = 0

        else:

            target = 0


        # ----------------------------------------------------
        # 현재 행의 정보
        # ----------------------------------------------------

        row = group.iloc[i]


        machine_data.append({

            # 확인용
            "현재시간": current_time,
            "충전기명": charger_name,

            # ==========================
            # 머신러닝 입력 X
            # ==========================

            "최근1일고장횟수": count_1d,
            "최근7일고장횟수": count_7d,
            "최근30일고장횟수": count_30d,

            "이전고장후시간": hours_since_error,

            "시간": current_time.hour,
            "요일": current_time.weekday(),
            "월": current_time.month,

            # 현재 발생한 고장 코드
            "고장코드번호": row["고장코드번호"],

            # ==========================
            # 정답 Y
            # ==========================

            "24시간내재고장": target
        })


# ============================================================
# 6. DataFrame으로 변환
# ============================================================

data = pd.DataFrame(machine_data)


print()
print("================================")
print("머신러닝 데이터")
print("================================")

print(data.head())


print()
print("전체 학습 데이터:", len(data))


print()
print("정답 개수")
print(data["24시간내재고장"].value_counts())


print()
print("정답 비율")
print(data["24시간내재고장"].value_counts(normalize=True))


# ============================================================
# 7. Feature 선택
# ============================================================

features = [

    "최근1일고장횟수",
    "최근7일고장횟수",
    "최근30일고장횟수",

    "이전고장후시간",

    "시간",
    "요일",
    "월",

    "고장코드번호"
]


# ============================================================
# 8. 시간순 Train / Test 분리
#
# 랜덤으로 섞으면 안 됨.
#
# 과거 80% → 학습
# 미래 20% → 테스트
# ============================================================

data = data.sort_values(
    "현재시간"
).reset_index(drop=True)


cut_index = int(len(data) * 0.8)

cut_time = data.iloc[cut_index]["현재시간"]


train = data[
    data["현재시간"] < cut_time
]

test = data[
    data["현재시간"] >= cut_time
]


print()
print("================================")
print("Train / Test")
print("================================")

print(
    "Train:",
    train["현재시간"].min(),
    "~",
    train["현재시간"].max()
)

print(
    "Test :",
    test["현재시간"].min(),
    "~",
    test["현재시간"].max()
)


# ============================================================
# 9. X / Y
# ============================================================

x_train = train[features]
y_train = train["24시간내재고장"]

x_test = test[features]
y_test = test["24시간내재고장"]


# ============================================================
# 10. 머신러닝 모델
#
# Random Forest
#
# Decision Tree 여러 개를 만들어서
# 결과를 종합하는 모델
# ============================================================

model = RandomForestClassifier(

    # Tree 300개
    n_estimators=300,

    # Tree가 너무 복잡해지는 것 제한
    max_depth=10,

    # leaf에 최소 데이터 5개
    min_samples_leaf=5,

    # 클래스 비율이 불균형할 때 보정
    class_weight="balanced",

    random_state=1234,

    # CPU 코어 모두 사용
    n_jobs=-1
)


# ============================================================
# 11. ★ 머신러닝 학습
# ============================================================

model.fit(
    x_train,
    y_train
)


# ============================================================
# 12. ★ 미래 데이터 예측
# ============================================================

y_pred = model.predict(
    x_test
)


# 확률도 구하기
y_probability = model.predict_proba(
    x_test
)[:, 1]


# ============================================================
# 13. 정확도
# ============================================================

train_score = model.score(
    x_train,
    y_train
)

test_score = model.score(
    x_test,
    y_test
)


print()
print("================================")
print("Random Forest 결과")
print("================================")

print(
    "Train 정확도:",
    round(train_score, 4)
)

print(
    "Test 정확도:",
    round(test_score, 4)
)


# ============================================================
# 14. Confusion Matrix
# ============================================================

print()
print("Confusion Matrix")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ============================================================
# 15. Precision / Recall / F1
# ============================================================

print()
print("Classification Report")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# 16. 어떤 데이터가 예측에 중요했는지
# ============================================================

importance = pd.Series(
    model.feature_importances_,
    index=features
)

importance = importance.sort_values(
    ascending=False
)


print()
print("================================")
print("Feature 중요도")
print("================================")

print(importance)


# ============================================================
# 17. 실제 미래 / 머신러닝 예측 비교
# ============================================================

result = test[
    [
        "현재시간",
        "충전기명"
    ]
].copy()


result["실제"] = y_test.values
result["예측"] = y_pred

# 24시간 내 재고장이 날 확률
result["고장확률"] = y_probability


print()
print("================================")
print("실제 / 예측")
print("================================")

print(
    result.head(30)
)


# ============================================================
# 18. 고장 위험이 높은 것부터 보기
# ============================================================

danger = result.sort_values(
    "고장확률",
    ascending=False
)


print()
print("================================")
print("고장 위험도 높은 충전기")
print("================================")

print(
    danger.head(20)
)

원본 데이터 개수: 31013
중복 제거 후: 29724

머신러닝 데이터
                 현재시간      충전기명  최근1일고장횟수  최근7일고장횟수  최근30일고장횟수      이전고장후시간  \
0 2025-09-09 07:45:34  KDN-0001         0         0          0  9999.000000   
1 2025-09-09 08:18:55  KDN-0001         1         1          1     0.555833   
2 2025-09-09 08:58:20  KDN-0001         2         2          2     0.656944   
3 2025-09-09 23:29:34  KDN-0001         3         3          3    14.520556   
4 2025-12-06 04:57:04  KDN-0001         0         0          0  2093.458333   

   시간  요일   월  고장코드번호  24시간내재고장  
0   7   1   9     205         1  
1   8   1   9     205         1  
2   8   1   9     205         1  
3  23   1   9     205         0  
4   4   5  12     908         1  

전체 학습 데이터: 29724

정답 개수
24시간내재고장
1    29054
0      670
Name: count, dtype: int64

정답 비율
24시간내재고장
1    0.977459
0    0.022541
Name: proportion, dtype: float64

Train / Test
Train: 2025-07-03 19:12:46 ~ 2026-01-26 03:45:11
Test : 2026-01-26 03:45:26 ~ 2026-09-03 09:27:52

Random 